In [1]:
pip install statsbombpy pandas openpyxl


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 443.3 kB/s eta 0:00:00 kB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.5/67.5 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: attrs
    Found existing installation: attrs 23.1.0
    Uninstalling attrs-23.1.0:
      Successfully uninstalled attrs-23.1.0
Note: you may need to restart the kernel to use updated packages.


In [1]:
from statsbombpy import sb
sb.competitions()

/home/sina/anaconda3/lib/python3.11/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-11-15T23:17:41.827093,2025-11-15T23:17:41.827093,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,None,None,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2026-05-12T21:18:08.827431,2026-05-02T02:07:18.902396,2026-05-02T02:07:18.902396,2026-05-12T21:18:08.827431
3,16,4,Europe,Champions League,male,False,False,2018/2019,2026-05-15T15:54:04.598614,2021-06-13T16:17:31.694,None,2026-05-15T15:54:04.598614
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,None,2024-02-13T02:35:28.134882
...,...,...,...,...,...,...,...,...,...,...,...,...
75,35,75,Europe,UEFA Europa League,male,False,False,1988/1989,2026-04-11T12:48:10.012987,2021-06-13T16:17:31.694,None,2026-04-11T12:48:10.012987
76,53,315,Europe,UEFA Women's Euro,female,False,True,2025,2026-04-27T22:02:42.690507,2026-04-27T22:03:28.087062,2026-04-27T22:03:28.087062,2026-04-27T22:02:42.690507
77,53,106,Europe,UEFA Women's Euro,female,False,True,2022,2026-05-05T03:03:04.199896,2026-05-05T03:05:32.480837,2026-05-05T03:05:32.480837,2026-05-05T03:03:04.199896
78,72,107,International,Women's World Cup,female,False,True,2023,2026-05-03T13:51:31.021141,2026-05-03T13:55:52.303219,2026-05-03T13:55:52.303219,2026-05-03T13:51:31.021141


In [6]:
# Cell 1: Environment Check & Pinned Imports
import warnings
import pandas as pd
from statsbombpy import sb

warnings.filterwarnings("ignore")

# Print versions for your report's reproducibility appendix
print(f"Pandas Version: {pd.__version__}")

Pandas Version: 2.1.4


In [7]:
# Cell 2: Competition Selection

# Fetch all open competition-season entries
competitions = sb.competitions()

# Filter for La Liga (Spain) to use as our pipeline backbone
la_liga_seasons = competitions[
    competitions["competition_name"] == "La Liga"
].sort_values(by="season_name")
print("Available La Liga Seasons:")
print(la_liga_seasons[["competition_id", "season_id", "season_name"]])

# Target 2015/2016 for our initial sandbox testing
target_season = la_liga_seasons[la_liga_seasons["season_name"] == "2015/2016"]
comp_id = int(target_season["competition_id"].values[0])
season_id = int(target_season["season_id"].values[0])

print(
    f"\nTargeting Competition ID: {comp_id} | Season ID: {season_id} (2015/2016)"
)

Available La Liga Seasons:
    competition_id  season_id season_name
57              11        278   1973/1974
56              11         37   2004/2005
55              11         38   2005/2006
54              11         39   2006/2007
53              11         40   2007/2008
52              11         41   2008/2009
51              11         21   2009/2010
50              11         22   2010/2011
49              11         23   2011/2012
48              11         24   2012/2013
47              11         25   2013/2014
46              11         26   2014/2015
45              11         27   2015/2016
44              11          2   2016/2017
43              11          1   2017/2018
42              11          4   2018/2019
41              11         42   2019/2020
40              11         90   2020/2021

Targeting Competition ID: 11 | Season ID: 27 (2015/2016)


In [8]:
# Cell 3: Pull a Sample Match

# Load all matches in the chosen season
matches_df = sb.matches(competition_id=comp_id, season_id=season_id)
print(f"Total matches found in season: {len(matches_df)}")

# Pick a single sample match_id to build out our sandbox features
sample_match_id = matches_df["match_id"].iloc[0]
sample_match_info = matches_df[matches_df["match_id"] == sample_match_id].iloc[
    0
]
print(
    f"Sample Match: {sample_match_info['home_team']} vs {sample_match_info['away_team']}"
)

# Pull raw events for this single match (around 4,000 rows)
events_df = sb.events(match_id=sample_match_id)
print(f"Loaded {len(events_df)} raw events for match_id {sample_match_id}.")
print("Available event columns sample:", list(events_df.columns[:10]))

Total matches found in season: 380
Sample Match: Real Madrid vs Sporting Gijón
Loaded 3947 raw events for match_id 3825739.
Available event columns sample: ['50_50', 'ball_receipt_outcome', 'ball_recovery_recovery_failure', 'block_deflection', 'block_offensive', 'carry_end_location', 'clearance_aerial_won', 'clearance_body_part', 'clearance_head', 'clearance_left_foot']


In [10]:
# Load the downloaded historical odds CSV
# We use '../' to step out of the Notebook folder and look into the data folder
odds_path = "../data/raw/odds/spain_la_liga_2015_2016.csv"

try:
    odds_df = pd.read_csv(odds_path)
    print("Successfully loaded historical odds data!")
    print(f"Total records in market data: {len(odds_df)}")

    # Extract unique team names to build our name alignment mapping (alias dictionary)
    sb_teams = sorted(matches_df["home_team"].unique())
    market_teams = sorted(odds_df["HomeTeam"].unique())

    print("\n--- Team Name Conventions Side-by-Side ---")
    # Using min length to avoid index errors if the sizes differ slightly
    for i in range(min(len(sb_teams), len(market_teams))):
        print(f"StatsBomb: '{sb_teams[i]}'  |  Market CSV: '{market_teams[i]}'")

except FileNotFoundError:
    print(
        f"🚨 Path Error: Could not find the file at {odds_path}.\n"
        "Double check that your notebook environment treats the 'Notebook' directory as its root!"
    )

Successfully loaded historical odds data!
Total records in market data: 380

--- Team Name Conventions Side-by-Side ---
StatsBomb: 'Athletic Club'  |  Market CSV: 'Ath Bilbao'
StatsBomb: 'Atlético Madrid'  |  Market CSV: 'Ath Madrid'
StatsBomb: 'Barcelona'  |  Market CSV: 'Barcelona'
StatsBomb: 'Celta Vigo'  |  Market CSV: 'Betis'
StatsBomb: 'Eibar'  |  Market CSV: 'Celta'
StatsBomb: 'Espanyol'  |  Market CSV: 'Eibar'
StatsBomb: 'Getafe'  |  Market CSV: 'Espanol'
StatsBomb: 'Granada'  |  Market CSV: 'Getafe'
StatsBomb: 'Las Palmas'  |  Market CSV: 'Granada'
StatsBomb: 'Levante UD'  |  Market CSV: 'La Coruna'
StatsBomb: 'Málaga'  |  Market CSV: 'Las Palmas'
StatsBomb: 'RC Deportivo La Coruña'  |  Market CSV: 'Levante'
StatsBomb: 'Rayo Vallecano'  |  Market CSV: 'Malaga'
StatsBomb: 'Real Betis'  |  Market CSV: 'Real Madrid'
StatsBomb: 'Real Madrid'  |  Market CSV: 'Sevilla'
StatsBomb: 'Real Sociedad'  |  Market CSV: 'Sociedad'
StatsBomb: 'Sevilla'  |  Market CSV: 'Sp Gijon'
StatsBomb: 'S

In [15]:
# =====================================================================
# CELL 5: FIXED TEAM ALIAS MAPPING & DE-VIGGED MARKET BASELINE
# =====================================================================

# 1. Heavily revised dictionary to match Football-Data.co.uk's exact strings
team_alias_map = {
    "Athletic Club": "Ath Bilbao",
    "Atlético Madrid": "Ath Madrid",
    "Celta Vigo": "Celta",
    "RC Deportivo La Coruña": "La Coruna",
    "Eibar": "Eibar",
    "Espanyol": "Espanol",
    "FC Barcelona": "Barcelona",
    "Barcelona": "Barcelona",
    "Getafe": "Getafe",
    "Granada": "Granada",
    "Las Palmas": "Las Palmas",
    "Levante UD": "Levante",
    "Málaga": "Malaga",
    "Rayo Vallecano": "Vallecano",
    "Real Betis": "Betis",
    "Real Madrid": "Real Madrid",
    "Real Sociedad": "Sociedad",
    "Sevilla": "Sevilla",
    "Sporting Gijón": "Sp Gijon",
    "Valencia": "Valencia",
    "Villarreal": "Villarreal",
}

# 2. Standardize dates in both dataframes to datetime.date objects
matches_df["clean_date"] = pd.to_datetime(matches_df["match_date"]).dt.date
odds_df["clean_date"] = pd.to_datetime(
    odds_df["Date"], format="%d/%m/%y"
).dt.date

# 3. Map StatsBomb teams to their corrected market aliases
matches_df["market_home"] = matches_df["home_team"].map(team_alias_map)
matches_df["market_away"] = matches_df["away_team"].map(team_alias_map)

# 4. Perform the join strictly on (date, home team, away team)
integrated_df = pd.merge(
    matches_df,
    odds_df[["clean_date", "HomeTeam", "AwayTeam", "B365H", "B365D", "B365A"]],
    left_on=["clean_date", "market_home", "market_away"],
    right_on=["clean_date", "HomeTeam", "AwayTeam"],
    how="inner",
)

print(f"StatsBomb matches total: {len(matches_df)}")
print(f"Successfully tagged with market odds: {len(integrated_df)}")
coverage_rate = (len(integrated_df) / len(matches_df)) * 100
print(f"Tagging Coverage Rate: {coverage_rate:.2f}%")

# 5. De-vig the market odds
integrated_df["implied_H"] = 1 / integrated_df["B365H"]
integrated_df["implied_D"] = 1 / integrated_df["B365D"]
integrated_df["implied_A"] = 1 / integrated_df["B365A"]

integrated_df["margin"] = (
    integrated_df["implied_H"]
    + integrated_df["implied_D"]
    + integrated_df["implied_A"]
)

integrated_df["market_prob_H"] = (
    integrated_df["implied_H"] / integrated_df["margin"]
)
integrated_df["market_prob_D"] = (
    integrated_df["implied_D"] / integrated_df["margin"]
)
integrated_df["market_prob_A"] = (
    integrated_df["implied_A"] / integrated_df["margin"]
)

StatsBomb matches total: 380
Successfully tagged with market odds: 380
Tagging Coverage Rate: 100.00%


In [16]:
# =====================================================================
# CELL 6: DIAGNOSING THE INTEGRATION FAILURES
# =====================================================================

# Find which StatsBomb matches did not make it into the integrated_df
merged_ids = integrated_df["match_id"].unique()
failed_matches = matches_df[~matches_df["match_id"].isin(merged_ids)]

print(f"Number of unmapped matches: {len(failed_matches)}")
print("\nSample of unmatched matches (Look closely at dates and teams):")
print(
    failed_matches[["clean_date", "home_team", "away_team", "match_id"]].head(
        10
    )
)

# Let's print a few rows from the odds file to compare their dates directly
print("\nSample of raw dates inside the Odds CSV file:")
print(odds_df[["Date", "HomeTeam", "AwayTeam"]].head(5))

Number of unmapped matches: 0

Sample of unmatched matches (Look closely at dates and teams):
Empty DataFrame
Columns: [clean_date, home_team, away_team, match_id]
Index: []

Sample of raw dates inside the Odds CSV file:
       Date    HomeTeam    AwayTeam
0  21/08/15      Malaga     Sevilla
1  22/08/15  Ath Madrid  Las Palmas
2  22/08/15     Espanol      Getafe
3  22/08/15   La Coruna    Sociedad
4  22/08/15   Vallecano    Valencia


In [17]:
# =====================================================================
# CELL 7: LEAKAGE-PROOF CHRONOLOGICAL TRAIN/TEST SPLIT
# =====================================================================

# Sort matches by date to preserve temporal order
integrated_df = integrated_df.sort_values(by="clean_date").reset_index(
    drop=True
)

# Define split sizes (e.g., 80% train/validation, 20% test)
split_idx = int(len(integrated_df) * 0.8)

train_val_matches = integrated_df.iloc[:split_idx]
test_matches = integrated_df.iloc[split_idx:]

print(f"Total Integrated Matches: {len(integrated_df)}")
print(
    f"Training/Validation Set: {len(train_val_matches)} matches (from {train_val_matches['clean_date'].min()} to {train_val_matches['clean_date'].max()})"
)
print(
    f"Test Set (Market Benchmark Evaluation): {len(test_matches)} matches (from {test_matches['clean_date'].min()} to {test_matches['clean_date'].max()})"
)

Total Integrated Matches: 380
Training/Validation Set: 304 matches (from 2015-08-21 to 2016-04-02)
Test Set (Market Benchmark Evaluation): 76 matches (from 2016-04-02 to 2016-05-15)


In [18]:
# =====================================================================
# CELL 8: DERIVING GROUND-TRUTH LABELS FOR TASKS C & R
# =====================================================================


def derive_labels(df):
    # Calculate signed goal margin (Home Score - Away Score)
    df["target_margin"] = df["home_score"] - df["away_score"]

    # Clip the margin to [-5, +5] as explicitly mandated by the project rules
    df["target_margin"] = df["target_margin"].clip(-5, 5)

    # Derive categorical outcome: Home Win (H), Draw (D), Away Win (A)
    df["target_outcome"] = "D"
    df.loc[df["target_margin"] > 0, "target_outcome"] = "H"
    df.loc[df["target_margin"] < 0, "target_outcome"] = "A"

    return df


# Apply the labeling logic to both data splits
train_val_matches = derive_labels(train_val_matches.copy())
test_matches = derive_labels(test_matches.copy())

print("Ground Truth Distribution in Training/Validation Set:")
print(train_val_matches["target_outcome"].value_counts(normalize=True))

print("\nSample Goal Margins and Outcomes:")
print(
    train_val_matches[
        ["home_team", "away_team", "home_score", "away_score", "target_margin", "target_outcome"]
    ].head()
)

Ground Truth Distribution in Training/Validation Set:
target_outcome
H    0.476974
A    0.276316
D    0.246711
Name: proportion, dtype: float64

Sample Goal Margins and Outcomes:
                home_team      away_team  home_score  away_score  \
0                  Málaga        Sevilla           0           0   
1          Rayo Vallecano       Valencia           0           0   
2                Espanyol         Getafe           1           0   
3  RC Deportivo La Coruña  Real Sociedad           0           0   
4         Atlético Madrid     Las Palmas           1           0   

   target_margin target_outcome  
0              0              D  
1              0              D  
2              1              H  
3              0              D  
4              1              H  


In [19]:
# =====================================================================
# CELL 9: LEEKAGE-PROOF ROLLING FORM FEATURE ENGINE
# =====================================================================


def compute_pre_match_features(target_matches, full_history_df, window=5):
    """Computes look-ahead leakage-proof rolling features for teams

    based strictly on matches completed BEFORE the target match date.
    """
    featured_records = []

    # Iterate through each match we want to predict
    for idx, match in target_matches.iterrows():
        match_date = match["clean_date"]
        home_team = match["home_team"]
        away_team = match["away_team"]

        # 1. Filter ALL history to matches completed STRICTLY BEFORE this match date
        prior_history = full_history_df[
            full_history_df["clean_date"] < match_date
        ]

        # 2. Extract Home Team historical metrics (both home and away appearances)
        home_past = prior_history[
            (prior_history["home_team"] == home_team)
            | (prior_history["away_team"] == home_team)
        ].sort_values(by="clean_date", ascending=False)

        # 3. Extract Away Team historical metrics (both home and away appearances)
        away_past = prior_history[
            (prior_history["home_team"] == away_team)
            | (prior_history["away_team"] == away_team)
        ].sort_values(by="clean_date", ascending=False)

        # Helper function to parse rolling stats safely
        def get_team_stats(past_games, team_name, n):
            if len(past_games) == 0:
                return 0.0, 0.0  # Default values for early season games

            # Take the last 'n' matches
            recent_games = past_games.head(n)

            goals_scored = []
            goals_conceded = []

            for _, g in recent_games.iterrows():
                if g["home_team"] == team_name:
                    goals_scored.append(g["home_score"])
                    goals_conceded.append(g["away_score"])
                else:
                    goals_scored.append(g["away_score"])
                    goals_conceded.append(g["home_score"])

            return sum(goals_scored) / len(goals_scored), sum(
                goals_conceded
            ) / len(goals_conceded)

        # Compute averages over the defined window (e.g., 5 matches)
        home_gf, home_ga = get_team_stats(home_past, home_team, window)
        away_gf, away_ga = get_team_stats(away_past, away_team, window)

        # Append features alongside identifiers and targets
        record = {
            "match_id": match["match_id"],
            "clean_date": match_date,
            "home_team": home_team,
            "away_team": away_team,
            # Engineered Features
            "home_rolling_gf": home_gf,  # Goals For
            "home_rolling_ga": home_ga,  # Goals Against
            "away_rolling_gf": away_gf,
            "away_rolling_ga": away_ga,
            # Baseline Odds & Labels
            "market_prob_H": match["market_prob_H"],
            "market_prob_D": match["market_prob_D"],
            "market_prob_A": match["market_prob_A"],
            "target_margin": match["target_margin"],
            "target_outcome": match["target_outcome"],
        }
        featured_records.append(record)

    return pd.DataFrame(featured_records)


# Build feature-rich matrices using integrated_df as the master history pool
print("Building training/validation feature matrix...")
X_y_train_val = compute_pre_match_features(
    train_val_matches, integrated_df, window=5
)

print("Building test evaluation feature matrix...")
X_y_test = compute_pre_match_features(test_matches, integrated_df, window=5)

print("\nEngineered Feature Set Sample (First 5 Rows):")
print(
    X_y_train_val[
        [
            "home_team",
            "away_team",
            "home_rolling_gf",
            "home_rolling_ga",
            "away_rolling_gf",
            "away_rolling_ga",
        ]
    ].head()
)

Building training/validation feature matrix...
Building test evaluation feature matrix...

Engineered Feature Set Sample (First 5 Rows):
                home_team      away_team  home_rolling_gf  home_rolling_ga  \
0                  Málaga        Sevilla              0.0              0.0   
1          Rayo Vallecano       Valencia              0.0              0.0   
2                Espanyol         Getafe              0.0              0.0   
3  RC Deportivo La Coruña  Real Sociedad              0.0              0.0   
4         Atlético Madrid     Las Palmas              0.0              0.0   

   away_rolling_gf  away_rolling_ga  
0              0.0              0.0  
1              0.0              0.0  
2              0.0              0.0  
3              0.0              0.0  
4              0.0              0.0  


In [20]:
# =====================================================================
# CELL 10: VERIFYING FEATURE ACCUMULATION LATER IN THE SEASON
# =====================================================================

print("Feature matrix sample from the middle of the season:")
print(
    X_y_train_val[
        [
            "clean_date",
            "home_team",
            "away_team",
            "home_rolling_gf",
            "home_rolling_ga",
        ]
    ].iloc[100:105]
)

Feature matrix sample from the middle of the season:
     clean_date       home_team               away_team  home_rolling_gf  \
100  2015-11-06      Las Palmas           Real Sociedad              0.4   
101  2015-11-07           Eibar                  Getafe              1.2   
102  2015-11-07  Rayo Vallecano                 Granada              1.4   
103  2015-11-07      Celta Vigo                Valencia              1.4   
104  2015-11-07      Levante UD  RC Deportivo La Coruña              0.2   

     home_rolling_ga  
100              2.2  
101              1.0  
102              2.2  
103              1.4  
104              2.6  


In [21]:
# =====================================================================
# CELL 11: IN-PLAY GAME SNAPSHOTTING WITH STRICT TIME-T ASSERTION
# =====================================================================


def generate_match_snapshot(match_events_df, snapshot_minute):
    """Filters a match event stream up to snapshot_minute

    and runs a strict assertion checking for look-ahead leakage.
    """
    # Copy data to prevent modifying the original dataframe
    events_pool = match_events_df.copy()

    # 1. Isolate events up to snapshot time t
    snapshot_df = events_pool[events_pool["minute"] <= snapshot_minute]

    # 2. MANDATORY TIME-T CUT ASSERTION (Directly required for Mid Defence 1)
    if not snapshot_df.empty:
        max_minute_found = snapshot_df["minute"].max()
        assert (
            max_minute_found <= snapshot_minute
        ), f"🚨 Leakage Detected! Found events from minute {max_minute_found} in a minute {snapshot_minute} snapshot."

    # 3. Compute baseline metrics up to time t
    # Get current score at this exact snapshot minute
    home_team_name = sample_match_info["home_team"]
    away_team_name = sample_match_info["away_team"]

    # Simple count of goals up to minute t
    # Note: StatsBomb events track goals via the 'shot_outcome' field or similar nested fields
    # For this sandbox, we will pull shot counts up to time t as a feature proxy
    total_shots = len(snapshot_df[snapshot_df["type"] == "Shot"])
    total_passes = len(snapshot_df[snapshot_df["type"] == "Pass"])

    snapshot_summary = {
        "snapshot_minute": snapshot_minute,
        "events_captured": len(snapshot_df),
        "cumulative_shots": total_shots,
        "cumulative_passes": total_passes,
    }

    return snapshot_summary, snapshot_df


# Let's simulate a snapshot at Minute 30 of our sample match
snapshot_time = 30
summary, filtered_events = generate_match_snapshot(events_df, snapshot_time)

print(f"--- Snapshot Test at Minute {snapshot_time} ---")
print(f"Total events filtered: {summary['events_captured']}")
print(f"Shots fired up to min 30: {summary['cumulative_shots']}")
print(f"Passes completed up to min 30: {summary['cumulative_passes']}")
print("\nLeakage verification check passed cleanly!")

--- Snapshot Test at Minute 30 ---
Total events filtered: 1250
Shots fired up to min 30: 12
Passes completed up to min 30: 379

Leakage verification check passed cleanly!
